<a href="https://colab.research.google.com/github/MarinaNasser/Nebras_R-D/blob/main/R%26D_medgemma_1_5_4b_it.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers
!pip install python-docx google-genai transformers torch pillow datasets

## Local Inference on GPU
Model page: https://huggingface.co/google/medgemma-1.5-4b-it

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/google/medgemma-1.5-4b-it)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# 1. Fetch the token and save it to a variable
hf_token = userdata.get('HF_TOKEN')

# 2. Pass that variable to the login function
login(token=hf_token)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="google/medgemma-1.5-4b-it")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'input_text': [{'role': 'user',
    'content': [{'type': 'image',
      'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
     {'type': 'text', 'text': 'What animal is on the candy?'}]}],
  'generated_text': [{'role': 'user',
    'content': [{'type': 'image',
      'url': 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG'},
     {'type': 'text', 'text': 'What animal is on the candy?'}]},
   {'role': 'assistant',
    'content': 'The animal on the candy is a **penguin**.\n'}]}]

In [ ]:
!unzip reports.zip

Archive:  reports.zip
   creating: reports/
   creating: reports/cr/
  inflating: reports/cr/Ankel Joint Normal.docx  
  inflating: reports/cr/Ankel.docx   
  inflating: reports/cr/barium swallow.docx  
  inflating: reports/cr/both calenous bones.docx  
  inflating: reports/cr/both feet.docx  
  inflating: reports/cr/both hips joints (2 views).docx  
  inflating: reports/cr/Both Knee Normal.docx  
  inflating: reports/cr/both knee.docx  
  inflating: reports/cr/Cervical spine.docx  
  inflating: reports/cr/cervical vertebrae.docx  
  inflating: reports/cr/cervical vertebrae1 (2).docx  
  inflating: reports/cr/cervical vertebrae1.docx  
  inflating: reports/cr/chest (2).docx  
  inflating: reports/cr/chest and heart.docx  
  inflating: reports/cr/Chest.docx   
  inflating: reports/cr/chest1.docx  
  inflating: reports/cr/dorsal vertebrae.docx  
  inflating: reports/cr/dorsal.docx  
  inflating: reports/cr/elbow joint.docx  
  inflating: reports/cr/EXAMINATION OF THE  LEFT   ELBOW.docx  

In [ ]:
import os
import glob
from docx import Document
from google import genai
from transformers import pipeline
import torch

# ==========================================
# 1. INITIALIZE API TOKENS & CLIENTS
# ==========================================
# Replace these with your actual tokens or retrieve via Colab userdata
GEMINI_API_KEY = userdata.get('gemini_api_key')
HF_TOKEN = hf_token

# Initialize the official Google GenAI SDK client
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def extract_text_from_docx(file_path):
    """Reads a .docx file and extracts all raw text."""
    doc = Document(file_path)
    full_text = []
    for para in doc.paragraphs:
        full_text.append(para.text)
    # Also extract text trapped inside tables if any exist
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                full_text.append(cell.text)
    return "\n".join(full_text)

def gemini_extract_findings(raw_text):
    """Uses Gemini to distill the raw report into clean bullet points."""
    prompt = f"""
    You are an expert medical data extraction assistant.
    Analyze the following raw text from a medical radiology report.
    Extract and list all clinical 'MR Findings' and the 'Opinion' section into clear, concise, and professional bullet points.
    Do not include administrative details (like patient name, dates, or doctor names).

    Raw Report Text:
    ---
    {raw_text}
    ---

    Bullet Points:
    """

    response = gemini_client.models.generate_content(
        model='gemini-3-flash-preview',
        contents=prompt
    )
    return response.text

from google.genai import types

def gemini_validate_report(findings, structured_report):
    """Uses Gemini to audit MedGemma's generated report with the JSON structure defined in the prompt."""

    prompt = f"""
    You are a senior medical director auditing an automated medical report generator.
    Evaluate how well the Generated Clinical Report covers the Original Extracted Findings.

    Original Extracted Findings:
    {findings}

    Generated Clinical Report:
    {structured_report}

    Critique the generation based on:
    1. Accuracy: Did the generated report introduce any hallucinations or medical errors?
    2. Completeness: Did it omit any critical findings from the original notes (e.g., nerve entrapment, scoliosis details)?

    Scoring Rules:
    - Assign a quantitative score out of 10.
    - If the score is 7 or above, set status to "PASSED".
    - If the score is below 7, set status to "NOT PASSED".

    You must return a valid JSON object matching the exact structure below. Do not add any markdown wrapping, text before, or text after the JSON object.

    Output Response Structure:
    {{
        "status": "PASSED or NOT PASSED",
        "score": 10,
        "accuracy_critique": "Your brief evaluation notes here regarding hallucinations or accuracy",
        "completeness_critique": "Your brief evaluation notes here regarding omitted findings"
    }}
    """

    try:
        response = gemini_client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                # Enforces that the model speaks exclusively in JSON
                response_mime_type="application/json"            )
        )
        return response.text

    except Exception as e:
        print(f"   [!] Gemini Validation API Error: {e}")
        return '{ "status": "ERROR", "score": 0, "accuracy_critique": "API Error", "completeness_critique": "API Error" }'


In [ ]:
# ==========================================
# 3. INITIALIZE MEDGEMMA PIPELINE (OPTIMIZED)
# ==========================================
print("Loading MedGemma model... (Optimized for text-generation)")

# Switching to "text-generation" strips out the unused vision processing overhead
medgemma_pipe = pipeline(
    "text-generation",
    model="google/medgemma-1.5-4b-it",
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

Loading MedGemma model... (Optimized for text-generation)


config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
import json
import os
import glob

# ==========================================
# 4. EXECUTE PIPELINE OVER A FOLDER (FIXED FOR HIDDEN MS WORD FILES)
# ==========================================
folder_path = "/content/reports/mri"
output_file_path = "/content/pipeline_results.json"

docx_files = glob.glob(os.path.join(folder_path, "*.docx"))

if not docx_files:
    print(f"No .docx files found in {folder_path}.")

# Initialize or load existing results
all_results = []
processed_files = set()

if os.path.exists(output_file_path):
    try:
        with open(output_file_path, "r", encoding="utf-8") as f:
            all_results = json.load(f)
            processed_files = {record["file_name"] for record in all_results if "file_name" in record}
            print(f"[i] Found existing progress file. {len(processed_files)} files will be skipped.")
    except json.JSONDecodeError:
        all_results = []

for file_path in docx_files:
    filename = os.path.basename(file_path)

    # 🌟 CRITICAL FIX: Skip temporary/hidden MS Word owner files
    if filename.startswith("~$"):
        print(f" -> Ignoring temporary file: '{filename}'")
        continue

    # Checkpoint: Skip processing if file already exists in our cached database
    if filename in processed_files:
        print(f" -> Skipping: '{filename}' (Already processed)")
        continue

    print(f"\n{'='*50}\nProcessing file: {filename}\n{'='*50}")

    # Step A: Read Docx
    raw_text = extract_text_from_docx(file_path)

    # Step B: Gemini Extracts Findings
    print("\n[Step 1/3] Gemini extracting findings...")
    extracted_findings = gemini_extract_findings(raw_text)

    if not extracted_findings:
        print(f"   [!] Skipping {filename} due to extraction failure.")
        continue

    print("\n--- Extracted Bullet Points ---")
    print(extracted_findings)

    # Step C: MedGemma Generates Medical Report (OPTIMIZED)
    print("\n[Step 2/3] MedGemma generating clinical report...")
    prompt_text = f"You are a clinical AI agent. Expand these raw radiological findings into a formalized, detailed medical consultation report, offering clinical context for each point:\n\n{extracted_findings}"

    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = medgemma_pipe.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    medgemma_output = medgemma_pipe(
        formatted_prompt,
        max_new_tokens=600, # Set to 600 tokens to ensure complex reports don't cut off
        return_full_text=False
    )
    generated_report = medgemma_output[0]['generated_text']

    print("\n--- MedGemma Generated Report ---")
    print(generated_report)

    # Step D: Gemini Validates and Critiques Output
    print("\n[Step 3/3] Gemini validating generated report quality...")
    validation_output = gemini_validate_report(extracted_findings, generated_report)

    # Parse the JSON string coming from your custom validation prompt
    try:
        audit_results = json.loads(validation_output)
    except json.JSONDecodeError:
        print("   [!] Failed to parse Gemini response as JSON. Creating fallback object.")
        audit_results = {
            "status": "PARSE_ERROR",
            "score": 0,
            "accuracy_critique": "Raw string fallback",
            "completeness_critique": validation_output
        }

    print("\n--- Gemini Quality Audit Result ---")
    print(f"Status: {audit_results.get('status')}")
    print(f"Score:  {audit_results.get('score')}/10")
    print(f"Accuracy Critique:     {audit_results.get('accuracy_critique')}")
    print(f"Completeness Critique: {audit_results.get('completeness_critique')}")

    # Step E: Construct Log Record and Save to External File
    record = {
        "file_name": filename,
        "raw_extracted_text": raw_text,
        "gemini_extracted_findings": extracted_findings,
        "medgemma_generated_report": generated_report,
        "audit_metrics": {
            "status": audit_results.get("status"),
            "score": audit_results.get("score"),
            "accuracy_critique": audit_results.get("accuracy_critique"),
            "completeness_critique": audit_results.get("completeness_critique")
        }
    }

    all_results.append(record)

    # Save incrementally on every loop pass so you don't lose data
    with open(output_file_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=4)

    print(f"\n[✓] Progress saved successfully to: {output_file_path}")

print(f"\n{'='*50}\nPipeline complete! Total records saved: {len(all_results)}")


Processing file: parotid gland_1.docx

[Step 1/3] Gemini extracting findings...


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



--- Extracted Bullet Points ---
### MR Findings
*   **Parotid Glands:** Bilateral, almost symmetrical mild enlargement. The glands show hyperintense foci on T1 and T2 weighted imaging with signal suppression on STIR (consistent with fatty content). A few tiny, enhancing intraparotid nodules are present, which are isointense on T1 and T2.
*   **Lymph Nodes:** Bilateral mildly enlarged upper and lower deep cervical and submental lymph nodes. The largest is located on the left side, measuring approximately 1.6 x 1.1 x 2.2 cm.
*   **Submandibular Glands:** Normal MR appearance.
*   **Thyroid and Parathyroid Glands:** Normal MR appearance of both thyroid lobes. No evidence of mass lesions at the parathyroid anatomical sites down to the level of the aortic arch.
*   **Airway and Larynx:** Patent oro-pharyngeal and laryngeal pathways. Vocal cords and laryngeal skeleton are intact.
*   **Vascular and Musculoskeletal:** Normal appearance of both carotid sheaths. Cervical musculature and interv

[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- MedGemma Generated Report ---
## Medical Consultation Report: Magnetic Resonance Imaging (MRI) of the Neck

**Patient Name:** [Patient Name - Placeholder]
**Date of Examination:** [Date - Placeholder]
**Referring Physician:** [Referring Physician - Placeholder]
**Report Date:** [Date - Placeholder]
**Reporting Physician:** [Your AI Agent Name - Placeholder]

**Clinical History:** [Provide brief clinical context if available, e.g., "Patient presents with bilateral neck swelling."]

**Imaging Modality:** Contrast-Enhanced MRI of the Neck

**Findings:**

**1. Parotid Glands:**
*   **Description:** Both parotid glands appear mildly enlarged bilaterally, demonstrating near symmetry.
*   **Signal Characteristics:** On T1-weighted imaging, the glands demonstrate a characteristic hyperintense signal, consistent with fat content. Similarly, on T2-weighted imaging, the glands show hyperintense signal.
*   **STIR Sequence:** On Short Tau Inversion Recovery (STIR) sequences, there is signal s

...


--- Extracted Bullet Points ---
**MR Findings**
*   **Acute Findings:** An oval-shaped area of increased T2 and FLAIR signal intensity with restricted diffusion is present in the right high fronto-parietal, para-sagittal subcortical region. No significant mass effect is observed.
*   **White Matter Changes:** 
    *   Bilateral deep periventricular patchy areas of increased T2 and FLAIR signal intensity.
    *   Multiple bilateral subcortical and deep periventricular small foci of abnormal T2 and FLAIR signals without mass effect.
    *   Tiny foci with similar signal patterns noted within the pons.
*   **Brain Volume/CSF Spaces:** Prominence of the ventricular system and extra-axial CSF spaces, including the cerebral cortical sulci, Sylvian fissures, and basal cisterns.
*   **Integumentary/Structural:** 
    *   Normal appearance of the cerebellum and sella turcica.
    *   No midline shift or extra-axial collections.
*   **Incidental Findings:** Mild bilateral maxillary sinusitis.



[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Extracted Bullet Points ---
**MR Findings**

*   **Progression of Known Lesion:** Significant interval increase in the size of the previously identified well-defined sellar, parasellar, and supra-sellar space-occupying lesion. It currently measures approximately 6.0 x 6.8 x 5.7 cm (previously 4.5 x 4.8 x 4.2 cm).
*   **Lesion Characteristics:** The mass exhibits intermediate T1 signal with foci suggestive of hemorrhage or calcification, and bright T2/FLAIR signals with evidence of blooming on SWAN (confirming calcification or blood products). It shows intense heterogeneous post-contrast enhancement with areas of central cystic degeneration/necrosis.
*   **Anatomical Extensions:**
    *   **Inferior:** Encroachment and invasion of the sphenoid sinus.
    *   **Lateral:** Encroachment upon both cavernous sinuses; total encasement of the left internal carotid artery (ICA) and partial encasement of the right ICA. Both arteries remain patent.
    *   **Superior:** Obliteration of the s

[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Extracted Bullet Points ---
Based on the medical report provided, here are the extracted clinical findings and the conclusion:

**MR Findings**
*   **Mediastinum/Thymus:** No mediastinal masses or thymic lesions detected.
*   **Lungs:** Both lungs are clear; no evidence of pulmonary infiltrations or mass lesions.
*   **Pleura:** No detectable pleural effusion or thickening on either side.
*   **Lymph Nodes:** No hilar or mediastinal lymphadenopathy visualized.
*   **Heart:** No gross cardiac abnormalities detected.
*   **Lung Bases:** Normal MRI appearance of the lung bases.

**Opinion / Conclusion**
*   Normal MRI study of the chest.
*   No mediastinal masses or thymic lesions detected.

[Step 2/3] MedGemma generating clinical report...

--- MedGemma Generated Report ---
## Clinical Consultation Report: Chest MRI

**Patient Information:** [Insert Patient Name, DOB, MRN, Date of Examination]

**Referring Physician:** [Insert Referring Physician Name, Specialty]

**Date of Report:*

In [ ]:
import json
import os
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement, parse_xml
from docx.oxml.ns import nsdecls, qn

def set_cell_background(cell, fill_hex):
    """Sets the background color of a table cell."""
    tcPr = cell._tc.get_or_add_tcPr()
    shd = parse_xml(f'<w:shd {nsdecls("w")} w:fill="{fill_hex}"/>')
    tcPr.append(shd)

def set_cell_margins(cell, top=100, bottom=100, left=150, right=150):
    """Sets internal padding (margins) for a table cell in twentieths of a point (dxa)."""
    tcPr = cell._tc.get_or_add_tcPr()
    tcMar = OxmlElement('w:tcMar')
    for m, val in [('top', top), ('bottom', bottom), ('left', left), ('right', right)]:
        node = OxmlElement(f'w:{m}')
        node.set(qn('w:w'), str(val))
        node.set(qn('w:type'), 'dxa')
        tcMar.append(node)
    tcPr.append(tcMar)

def create_doctor_review_docx(json_path, output_docx_path):
    if not os.path.exists(json_path):
        print(f"[!] Error: Reference file '{json_path}' not found.")
        return

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    doc = Document()

    # Define Core Document Styles
    style_normal = doc.styles['Normal']
    style_normal.font.name = 'Arial'
    style_normal.font.size = Pt(10.5)
    style_normal.font.color.rgb = RGBColor(0x33, 0x33, 0x33)

    # Document Header
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run_title = title.add_run("CLINICAL PIPELINE VALIDATION SHEET")
    run_title.font.size = Pt(18)
    run_title.font.bold = True
    run_title.font.color.rgb = RGBColor(0x1B, 0x36, 0x5D) # Deep Clinical Blue

    subtitle = doc.add_paragraph()
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run_sub = subtitle.add_run(f"Total Cases Logged: {len(data)}  |  Generated for Expert Medical Review")
    run_sub.font.size = Pt(10)
    run_sub.font.italic = True
    run_sub.font.color.rgb = RGBColor(0x66, 0x66, 0x66)

    doc.add_paragraph("_" * 70).paragraph_format.space_after = Pt(24)

    # Iterate over each case file record
    for idx, case in enumerate(data, 1):
        filename = case.get("file_name", f"Case_{idx}")

        # Heading 1: Case Separator
        h1 = doc.add_paragraph()
        h1.paragraph_format.space_before = Pt(18)
        h1.paragraph_format.space_after = Pt(6)
        r_h1 = h1.add_run(f"Case #{idx}: {filename}")
        r_h1.font.size = Pt(14)
        r_h1.font.bold = True
        r_h1.font.color.rgb = RGBColor(0x1B, 0x36, 0x5D)

        # Summary Status Table Callout Box
        metrics = case.get("audit_metrics", {})
        status = metrics.get("status", "UNKNOWN")
        score = metrics.get("score", 0)

        box_table = doc.add_table(rows=1, cols=2)
        box_table.autofit = False
        box_table.columns[0].width = Pt(150)
        box_table.columns[1].width = Pt(350)

        cell_status, cell_critique = box_table.cell(0, 0), box_table.cell(0, 1)

        # Color match depending on pass status
        bg_color = "E2F0D9" if status == "PASSED" else "FCE4D6"
        set_cell_background(cell_status, bg_color)
        set_cell_background(cell_critique, "F2F2F2")
        set_cell_margins(cell_status, top=140, bottom=140, left=180, right=180)
        set_cell_margins(cell_critique, top=140, bottom=140, left=180, right=180)

        p_stat = cell_status.paragraphs[0]
        r_stat = p_stat.add_run(f"AI AUDIT: {status}\nScore: {score}/10")
        r_stat.font.bold = True
        r_stat.font.size = Pt(11)
        r_stat.font.color.rgb = RGBColor(0x38, 0x57, 0x23) if status == "PASSED" else RGBColor(0xC0, 0x00, 0x00)

        p_crit = cell_critique.paragraphs[0]
        p_crit.add_run("Accuracy Notes: ").font.bold = True
        p_crit.add_run(f"{metrics.get('accuracy_critique', 'N/A')}\n")
        p_crit.add_run("Omission Notes: ").font.bold = True
        p_crit.add_run(f"{metrics.get('completeness_critique', 'N/A')}")
        p_crit.runs[0].font.size = Pt(9.5)

        doc.add_paragraph().paragraph_format.space_after = Pt(8)

        # Section A: Extracted Findings
        h2_a = doc.add_paragraph()
        r_h2_a = h2_a.add_run("1. Distilled Target Findings (Gemini)")
        r_h2_a.font.size = Pt(11)
        r_h2_a.font.bold = True
        r_h2_a.font.color.rgb = RGBColor(0x41, 0x71, 0x9C)

        p_find = doc.add_paragraph(case.get("gemini_extracted_findings", ""))
        p_find.paragraph_format.left_indent = Pt(12)
        p_find.paragraph_format.space_after = Pt(12)

        # Section B: MedGemma Draft Report
        h2_b = doc.add_paragraph()
        r_h2_b = h2_b.add_run("2. Expanded Consultation Report Draft (MedGemma)")
        r_h2_b.font.size = Pt(11)
        r_h2_b.font.bold = True
        r_h2_b.font.color.rgb = RGBColor(0x41, 0x71, 0x9C)

        p_rep = doc.add_paragraph(case.get("medgemma_generated_report", ""))
        p_rep.paragraph_format.left_indent = Pt(12)
        p_rep.paragraph_format.space_after = Pt(16)

        # Section C: Doctor Notes Sign-off Area
        p_sign = doc.add_paragraph()
        p_sign.paragraph_format.space_before = Pt(8)
        p_sign.paragraph_format.space_after = Pt(24)
        r_sign = p_sign.add_run("Physician Validation Sign-off:\n[  ] APPROVED UNCHANGED    [  ] REJECTED WITH CORRECTIONS\n\nNotes / Corrections: _____________________________________________________________________\n\nSignature: ___________________________                         Date: ________________________")
        r_sign.font.size = Pt(9.5)
        r_sign.font.color.rgb = RGBColor(0x7F, 0x7F, 0x7F)

        # Separator line between cases
        if idx < len(data):
            doc.add_page_break()

    doc.save(output_docx_path)
    print(f"[✓] Validation sheets successfully written to Word file: '{output_docx_path}'")

# Execute the creation function
create_doctor_review_docx("/content/pipeline_results.json", "/content/Doctor_Review_Sheet.docx")

[✓] Validation sheets successfully written to Word file: '/content/Doctor_Review_Sheet.docx'


In [ ]:
import json
import os

def display_pipeline_summary(json_path):
    if not os.path.exists(json_path):
        print(f"[!] No results file found at '{json_path}' yet. Run your processing loop first!")
        return

    with open(json_path, "r", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            print("[!] Error: The results file is corrupted or empty.")
            return

    total_docs = len(data)
    passed_count = 0
    not_passed_count = 0
    error_count = 0

    scores = []
    failed_files = []

    for case in data:
        filename = case.get("file_name", "Unknown File")
        metrics = case.get("audit_metrics", {})
        status = metrics.get("status", "UNKNOWN").upper()
        score = metrics.get("score", 0)

        if "PASSED" in status:
            passed_count += 1
            scores.append(score)
        elif "NOT PASSED" in status or "REVISION" in status:
            not_passed_count += 1
            scores.append(score)
            failed_files.append((filename, score, metrics.get("completeness_critique", "No critique notes.")))
        else:
            error_count += 1

    # Calculations
    processed_valid = passed_count + not_passed_count
    avg_score = sum(scores) / len(scores) if scores else 0.0
    pass_rate = (passed_count / processed_valid * 100) if processed_valid > 0 else 0.0

    # Print Summary Dashboard
    print(f"{'='*50}")
    print(f"         CLINICAL PIPELINE METRICS DASHBOARD        ")
    print(f"{'='*50}")
    print(f" 📂 Total Documents Processed : {total_docs}")
    print(f" ✅ Total Passed Cases        : {passed_count}")
    print(f" ❌ Total Not Passed Cases    : {not_passed_count}")

    if error_count > 0:
        print(f" ⚠️  System Parsing Errors     : {error_count}")

    print(f"{'-'*50}")
    print(f" 📈 Pipeline Pass Rate        : {pass_rate:.1f}%")
    print(f" 🎯 Average Clinical Score    : {avg_score:.2f} / 10")
    print(f"{'='*50}")

    # Flagged Action Items for the Doctor/Engineer
    if failed_files:
        print("\n🚨 CASES REQUIRING REVIEW (Score < 7):")
        for name, score, critique in failed_files:
            print(f"  • File: {name} (Score: {score}/10)")
            print(f"    Critique: {critique}\n")
    else:
        print("\n🎉 Exceptional Performance: All processed records passed audit guidelines!")

# Execute the metrics calculator
display_pipeline_summary("/content/pipeline_results.json")

         CLINICAL PIPELINE METRICS DASHBOARD        
 📂 Total Documents Processed : 77
 ✅ Total Passed Cases        : 77
 ❌ Total Not Passed Cases    : 0
--------------------------------------------------
 📈 Pipeline Pass Rate        : 100.0%
 🎯 Average Clinical Score    : 5.44 / 10

🎉 Exceptional Performance: All processed records passed audit guidelines!


# Specific domain test


In [ ]:
# ==========================================
# TEST CASES WITHOUT EXAM NAMES
# ==========================================
test_cases = [
    {
        "name": "Test Case 1 (Hidden Domain: Cervical Spine)",
        "findings": "Central disc extrusion at C5-6 with cranial migration causing 4mm indentation of the ventral aspect of the cord and severe narrowing of the right exiting neural foramen."
    },
    {
        "name": "Test Case 2 (Hidden Domain: Lumbar Spine)",
        "findings": "Diffuse posterior disc bulge at L5-S1 effacing the anterior epidural fat pad, flattening the thecal sac, with partial entrapment of the left S1 nerve root within the lateral recess."
    },
    {
        "name": "Test Case 3 (Hidden Domain: Head / Brain / Auditory)",
        "findings": "A well-defined 1.2 cm enhancing mass lesion is noted within the left cerebellopontine angle, extending into the internal auditory canal, widening it. Normal appearance of the brainstem."
    }
]

print("🔬 Running Domain Latency Tests on MedGemma...\n")

for case in test_cases:
    print(f"{'='*60}")
    print(f"🏃 {case['name']}")
    print(f"📥 Input Provided to Model:\n   \"{case['findings']}\"")
    print(f"{'-'*60}")

    # Wrap in the conversational instruction format
    prompt_text = (
        f"You are a clinical AI agent. Expand these raw radiological findings into a formalized, "
        f"detailed medical consultation report. Ensure you explicitly identify the anatomical "
        f"examination domain in your report header based on the text context:\n\n{case['findings']}"
    )

    messages = [{"role": "user", "content": prompt_text}]
    formatted_prompt = medgemma_pipe.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Generate response
    output = medgemma_pipe(
        formatted_prompt,
        max_new_tokens=300,
        return_full_text=False
    )

    print("📤 MedGemma Output:")
    print(output[0]['generated_text'].strip())
    print(f"{'='*60}\n")

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔬 Running Domain Latency Tests on MedGemma...

🏃 Test Case 1 (Hidden Domain: Cervical Spine)
📥 Input Provided to Model:
   "Central disc extrusion at C5-6 with cranial migration causing 4mm indentation of the ventral aspect of the cord and severe narrowing of the right exiting neural foramen."
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Output:
Okay, here is the formalized medical consultation report based on the provided radiological findings:

**Medical Consultation Report**

**Patient Information:** [Patient Name/ID - *assuming placeholder*]
**Date of Report:** [Date]

**Clinical Presentation:** [Briefly mention reason for imaging, e.g., Low back pain, radiculopathy, etc. - *assuming placeholder*]

**Radiological Findings:**

**Anatomical Examination Domain:** **Cervical Spine**

**Imaging Modality:** [Specify Modality, e.g., MRI, CT - *assuming MRI*]

**Findings:**

1.  **Disc Level:** C5-6
2.  **Disc Pathology:** There is evidence of a central disc extrusion at the C5-6 level.
3.  **Cranial Migration:** The extruded disc material is migrating cranially (superiorly) towards the spinal canal.
4.  **Cord Compression:** The cranial migration of the disc material is causing significant indentation of the ventral (anterior) aspect of the cervical spinal cord. The degree of cord indentation is estimated at ap

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Output:
Okay, here is the formalized medical consultation report based on the provided radiological findings:

**Medical Consultation Report**

**Patient Information:** (Placeholder - Specific patient details would be included here in a real scenario)

**Date of Examination:** (Placeholder - Date the imaging was performed)

**Imaging Modality:** Magnetic Resonance Imaging (MRI) of the Lumbar Spine

**Anatomical Examination Domain:** Lumbar Spine

**Clinical Indication:** (Placeholder - Reason for the MRI, e.g., low back pain, radiculopathy)

**Findings:**

*   **Lumbar Spine:**
    *   **Posterior Disc Bulge:** A diffuse posterior disc bulge is observed at the L5-S1 intervertebral disc level.
    *   **Epidural Fat:** The bulge is effacing (compressing) the anterior epidural fat pad.
    *   **Thecal Sac:** The bulge is causing flattening of the thecal sac (spinal canal).
    *   **Nerve Root Compression:** There is partial entrapment of the left S1 nerve root within the lat

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import torch
import random

# ==========================================
# 1. LOAD MODEL (MULTIMODAL VERSION)
# ==========================================
# Re-initializing using "image-text-to-text" to process pixel matrices
print("Loading Multimodal MedGemma 1.5 4B...")
medgemma_vqa = pipeline(
    "image-text-to-text",
    model="google/medgemma-1.5-4b-it",
    token=userdata.get('HF_TOKEN'), # Uses your updated Colab secret name
    torch_dtype=torch.bfloat16,
    device_map="auto"
)


Loading Multimodal MedGemma 1.5 4B...


config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [ ]:
import json
import os
import random
from datasets import load_dataset

# ==========================================
# 2. STREAM RADIOLOGY IMAGES (UNIFORM DISTRIBUTION)
# ==========================================
print("\nStreaming samples from hf-vision/chest-xray-pneumonia...")
dataset = load_dataset("hf-vision/chest-xray-pneumonia", split="test", streaming=True)

# Define your uniform distribution target
samples_per_class = 50

# Separate buckets to accumulate images
normal_samples = []
pneumonia_samples = []

print(f"⌛ Collecting a balanced set ({samples_per_class} per class)...")

# Iterate through the stream dynamically
for item in dataset:
    label = item['label']

    # Class 0: Normal
    if label == 0 and len(normal_samples) < samples_per_class:
        normal_samples.append(item)

    # Class 1: Pneumonia
    elif label == 1 and len(pneumonia_samples) < samples_per_class:
        pneumonia_samples.append(item)

    # Break early the second both buckets are perfectly filled
    if len(normal_samples) == samples_per_class and len(pneumonia_samples) == samples_per_class:
        break

# Combine both subsets into a single uniform evaluation list
test_samples = normal_samples + pneumonia_samples

# Shuffle the combined list so MedGemma doesn't get all Normals followed by all Pneumonias
random.shuffle(test_samples)

# Map class integers to text labels for console visibility and accuracy checking
label_map = {0: "Normal", 1: "Pneumonia"}

print(f"✅ Successfully loaded {len(test_samples)} images with a perfect 50/50 uniform distribution!")


# ==========================================
# 3. RUN MULTIMODAL INFERENCE & COMPUTE ACCURACY
# ==========================================
output_vqa_path = "/content/radiology_classification_results.json"

print("\n🔬 Starting Multimodal Classification Accuracy Evaluation...\n")

vqa_results = []
correct_predictions = 0

for idx, item in enumerate(test_samples, 1):
    pil_image = item['image']
    ground_truth = label_map.get(item['label'], "Unknown")

    if pil_image.mode != 'RGB':
        pil_image = pil_image.convert('RGB')

    print(f"{'='*60}")
    print(f"🏃 Diagnostic Test Case #{idx} / {len(test_samples)}")
    print(f"📋 Dataset Ground Truth Label: {ground_truth}")
    print(f"{'-'*60}")

    # Restructuring prompt to constrain MedGemma to a strict classification output choice
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": pil_image
                },
                {
                    "type": "text",
                    "text": (
                        "Analyze this chest radiograph (CXR). You must perform binary classification. "
                        "Determine if the image shows signs of Pneumonia or if it is Normal.\n"
                        "Do not write descriptive findings or paragraphs. Your output must contain "
                        "exclusively one word, either 'Normal' or 'Pneumonia'."
                    )
                }
            ]
        }
    ]

    # Run generation with low max_new_tokens to prevent paragraph generation
    output = medgemma_vqa(
        text=messages,
        max_new_tokens=10
    )

    # Extract the string content from the returned chat message structure
    chat_history = output[0]['generated_text']
    generated_assessment = chat_history[-1]['content'].strip()

    # Clean up output parsing to ensure robust classification matching
    prediction = "Unknown"
    lower_assessment = generated_assessment.lower()
    if "pneumonia" in lower_assessment:
        prediction = "Pneumonia"
    elif "normal" in lower_assessment:
        prediction = "Normal"

    # Check accuracy matching
    is_correct = (prediction == ground_truth)
    if is_correct:
        correct_predictions += 1

    print(f"📤 MedGemma Predicted Output : {generated_assessment}")
    print(f"🎯 Matched Assessment Class  : {prediction} (Match: {is_correct})")
    print(f"{'='*60}\n")

    # Create a structured record including the clean matched class evaluation
    case_record = {
        "test_case_number": idx,
        "ground_truth_label": ground_truth,
        "medgemma_raw_output": generated_assessment,
        "parsed_prediction": prediction,
        "is_correct": is_correct
    }

    vqa_results.append(case_record)

    # Save incrementally to disk
    with open(output_vqa_path, "w", encoding="utf-8") as f:
        json.dump(vqa_results, f, ensure_ascii=False, indent=4)

# ==========================================
# 4. FINAL ACCURACY SUMMARY REPORT
# ==========================================
total_evaluated = len(vqa_results)
accuracy_percentage = (correct_predictions / total_evaluated) * 100 if total_evaluated > 0 else 0

print(f"\n{'='*60}")
print(f"     FINAL MEDGEMMA BINARY CLASSIFICATION METRICS     ")
print(f"{'='*60}")
print(f" 📂 Total Images Classified : {total_evaluated}")
print(f" ✅ Correct Predictions      : {correct_predictions}")
print(f" ❌ Incorrect Predictions    : {total_evaluated - correct_predictions}")
print(f"{'-'*60}")
print(f" 📈 CALCULATED ACCURACY RATE : {accuracy_percentage:.2f}%")
print(f"{'='*60}")
print(f"Structured dataset validation evaluation log successfully finalized at: {output_vqa_path}")


Streaming samples from hf-vision/chest-xray-pneumonia...
⌛ Collecting a balanced set (50 per class)...


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


✅ Successfully loaded 100 images with a perfect 50/50 uniform distribution!

🔬 Starting Multimodal Classification Accuracy Evaluation...

🏃 Diagnostic Test Case #1 / 100
📋 Dataset Ground Truth Label: Normal
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Predicted Output : Normal
🎯 Matched Assessment Class  : Normal (Match: True)

🏃 Diagnostic Test Case #2 / 100
📋 Dataset Ground Truth Label: Normal
------------------------------------------------------------


...

📤 MedGemma Predicted Output : Normal
🎯 Matched Assessment Class  : Normal (Match: True)

🏃 Diagnostic Test Case #98 / 100
📋 Dataset Ground Truth Label: Normal
------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Predicted Output : Normal
🎯 Matched Assessment Class  : Normal (Match: True)

🏃 Diagnostic Test Case #99 / 100
📋 Dataset Ground Truth Label: Normal
------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Predicted Output : Normal
🎯 Matched Assessment Class  : Normal (Match: True)

🏃 Diagnostic Test Case #100 / 100
📋 Dataset Ground Truth Label: Pneumonia
------------------------------------------------------------
📤 MedGemma Predicted Output : Normal
🎯 Matched Assessment Class  : Normal (Match: False)


     FINAL MEDGEMMA BINARY CLASSIFICATION METRICS     
 📂 Total Images Classified : 100
 ✅ Correct Predictions      : 73
 ❌ Incorrect Predictions    : 27
------------------------------------------------------------
 📈 CALCULATED ACCURACY RATE : 73.00%
Structured dataset validation evaluation log successfully finalized at: /content/radiology_classification_results.json


In [ ]:
import pandas as pd

df = pd.read_json(output_vqa_path)
# This displays a clean matrix of your labels alongside the clinical text
print(df[['test_case_number', 'ground_truth_label', 'medgemma_raw_output','parsed_prediction','is_correct']])

    test_case_number ground_truth_label medgemma_raw_output parsed_prediction  \
0                  1             Normal              Normal            Normal   
1                  2             Normal              Normal            Normal   
2                  3             Normal              Normal            Normal   
3                  4             Normal              Normal            Normal   
4                  5          Pneumonia              Normal            Normal   
..               ...                ...                 ...               ...   
95                96          Pneumonia           Pneumonia         Pneumonia   
96                97             Normal              Normal            Normal   
97                98             Normal              Normal            Normal   
98                99             Normal              Normal            Normal   
99               100          Pneumonia              Normal            Normal   

    is_correct  
0         

In [ ]:
print(f"The accuracy of the predictions is: {accuracy_percentage:.2f}%")

The accuracy of the predictions is: 73.00%


In [ ]:
!unzip /content/hyper-kvasir-segmented-images.zip

Archive:  /content/hyper-kvasir-segmented-images.zip
  inflating: segmented-images/bounding-boxes.json  
   creating: segmented-images/images/
  inflating: segmented-images/images/0004a718-546c-41c2-9c69-c4685093a039.jpg  
  inflating: segmented-images/images/0017b7c7-90f8-4de2-8723-1d87e5c58317.jpg  
  inflating: segmented-images/images/0048d8c5-b59d-461c-9834-f44a727e191d.jpg  
  inflating: segmented-images/images/00f98835-8fd8-43fe-960d-f4a1159c30f1.jpg  
  inflating: segmented-images/images/01503109-d81f-404e-a919-31de1aceb6b9.jpg  
  inflating: segmented-images/images/0198bc50-169e-456c-a56b-970a7f7d23b7.jpg  
  inflating: segmented-images/images/01c95ee8-79cc-46f4-a6a0-834d8a6045d4.jpg  
  inflating: segmented-images/images/01dbf397-c0fc-40e7-9b04-86de70781c9a.jpg  
  inflating: segmented-images/images/01dcc13a-85ab-4d6e-b42b-5d631e076ac2.jpg  
  inflating: segmented-images/images/01e6cfd7-bdb6-4929-9fc0-9a7dd2d3c7f7.jpg  
  inflating: segmented-images/images/026e9707-e74d-434f-9

In [ ]:
import json
import os
import random
from PIL import Image # Import PIL for image loading
import glob # Import glob to find image files

# ==========================================
# 2. LOAD ENDOSCOPY IMAGES FROM LOCAL DISK
# ==========================================
print("\nLoading samples from local 'segmented-images' directory...")

# Load bounding-boxes.json to get labels
annotations_path = "./segmented-images/bounding-boxes.json"
with open(annotations_path, "r") as f:
    annotations = json.load(f)

# FIX: Map top-level keys as filenames and extract labels from nested 'bbox' array
image_to_label = {}
for img_id, data in annotations.items():
    filename = f"{img_id}.jpg"  # Reconstruct the expected filename
    if "bbox" in data and len(data["bbox"]) > 0:
        image_to_label[filename] = data["bbox"][0]["label"]
    else:
        # Fallback if an entry has no bounding boxes
        image_to_label[filename] = "Unknown"

# Get all image paths
image_dir = "./segmented-images/images/"
all_image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))

# Prepare the dataset structure similar to Hugging Face dataset for consistency
local_dataset_items = []
for img_path in all_image_paths:
    filename = os.path.basename(img_path)
    if filename in image_to_label:
        try:
            img = Image.open(img_path).convert('RGB')
            local_dataset_items.append({
                'image': img,
                'label': image_to_label[filename]
            })
        except Exception as e:
            print(f"[!] Error loading image {filename}: {e}")

# Define your uniform distribution target (e.g., 25 Normal, 25 Polyps)
samples_per_class = 25

normal_samples = []
polyp_samples = []

print(f"⌛ Collecting a balanced set ({samples_per_class} per class)...")

# Iterate through the local dataset items
for item in local_dataset_items:
    label_text = str(item.get('label', '')).lower()

    # Target Class 1: Normal Colon Mucosa
    if "normal-colon-mucosa" in label_text and len(normal_samples) < samples_per_class:
        normal_samples.append(item)

    # Target Class 2: Polyps (Pathological Abnormality)
    elif "polyp" in label_text and len(polyp_samples) < samples_per_class: # HyperKvasir uses 'polyp' not 'polyps' in the JSON
        polyp_samples.append(item)

    # Break the moment both distinct diagnostic buckets are filled
    if len(normal_samples) == samples_per_class and len(polyp_samples) == samples_per_class:
        break

# Combine and shuffle subsets
test_samples = normal_samples + polyp_samples
random.shuffle(test_samples)

print(f"✅ Loaded {len(test_samples)} endoscopy images with a perfect 50/50 uniform distribution!")


# ==========================================
# 3. RUN MULTIMODAL INFERENCE & COMPUTE ACCURACY
# ==========================================
output_vqa_path = "/content/endoscopy_classification_results.json"

print("\n🔬 Starting Multimodal Endoscopy Classification Accuracy Evaluation...\n")

vqa_results = []
correct_predictions = 0

for idx, item in enumerate(test_samples, 1):
    pil_image = item['image']

    # Extract structural label strings to match against prediction tokens
    raw_label = item.get('label', 'Unknown')
    ground_truth = "Normal" if "normal" in raw_label.lower() else "Polyp"

    # No need to convert to RGB here, already done when loading image

    print(f"{'='*60}")
    print(f"🏃 Diagnostic Test Case #{idx} / {len(test_samples)}")
    print(f"📋 Dataset Ground Truth Label: {ground_truth} ({raw_label})")
    print(f"{'-'*60}")

    # Structure prompt to strictly constrain MedGemma to an endoscopy binary decision
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": pil_image
                },
                {
                    "type": "text",
                    "text": (
                        "Analyze this gastrointestinal endoscopy image. You must perform binary classification. "
                        "Determine if the tissue displays signs of a Polyp or if it is Normal mucosa.\n"
                        "Do not write descriptive findings or paragraphs. Your output must contain "
                        "exclusively one word, either 'Normal' or 'Polyp'."
                    )
                }
            ]
        }
    ]

    # Generate classification token output
    output = medgemma_vqa(
        text=messages,
        max_new_tokens=10
    )

    # Extract and strip text from message response payload
    chat_history = output[0]['generated_text']
    generated_assessment = chat_history[-1]['content'].strip()

    # Map predictions to strict eval boundaries
    prediction = "Unknown"
    lower_assessment = generated_assessment.lower()
    if "polyp" in lower_assessment:
        prediction = "Polyp"
    elif "normal" in lower_assessment:
        prediction = "Normal"

    # Track metrics
    is_correct = (prediction == ground_truth)
    if is_correct:
        correct_predictions += 1

    print(f"📤 MedGemma Predicted Output : {generated_assessment}")
    print(f"🎯 Matched Assessment Class  : {prediction} (Match: {is_correct})")
    print(f"{'='*60}\n")

    # Log structured metrics object
    case_record = {
        "test_case_number": idx,
        "ground_truth_label": ground_truth,
        "raw_label_name": raw_label,
        "medgemma_raw_output": generated_assessment,
        "parsed_prediction": prediction,
        "is_correct": is_correct
    }

    vqa_results.append(case_record)

    # Save incrementally to disk
    with open(output_vqa_path, "w", encoding="utf-8") as f:
        json.dump(vqa_results, f, ensure_ascii=False, indent=4)

# ==========================================
# 4. FINAL ACCURACY SUMMARY REPORT
# ==========================================
total_evaluated = len(vqa_results)
accuracy_percentage = (correct_predictions / total_evaluated) * 100 if total_evaluated > 0 else 0

print(f"\n{'='*60}")
print(f"     FINAL MEDGEMMA ENDOSCOPY CLASSIFICATION METRICS     ")
print(f"{'='*60}")
print(f" 📂 Total Images Classified : {total_evaluated}")
print(f" ✅ Correct Predictions      : {correct_predictions}")
print(f" ❌ Incorrect Predictions    : {total_evaluated - correct_predictions}")
print(f"{'-'*60}")
print(f" 📈 CALCULATED ACCURACY RATE : {accuracy_percentage:.2f}%\n")
print(f"{'='*60}")


Loading samples from local 'segmented-images' directory...


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


⌛ Collecting a balanced set (25 per class)...
✅ Loaded 25 endoscopy images with a perfect 50/50 uniform distribution!

🔬 Starting Multimodal Endoscopy Classification Accuracy Evaluation...

🏃 Diagnostic Test Case #1 / 25
📋 Dataset Ground Truth Label: Polyp (polyp)
------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Predicted Output : Polyp
🎯 Matched Assessment Class  : Polyp (Match: True)

🏃 Diagnostic Test Case #2 / 25
📋 Dataset Ground Truth Label: Polyp (polyp)
------------------------------------------------------------


...

📤 MedGemma Predicted Output : Polyp
🎯 Matched Assessment Class  : Polyp (Match: True)

🏃 Diagnostic Test Case #23 / 25
📋 Dataset Ground Truth Label: Polyp (polyp)
------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Predicted Output : Polyp
🎯 Matched Assessment Class  : Polyp (Match: True)

🏃 Diagnostic Test Case #24 / 25
📋 Dataset Ground Truth Label: Polyp (polyp)
------------------------------------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📤 MedGemma Predicted Output : Polyp
🎯 Matched Assessment Class  : Polyp (Match: True)

🏃 Diagnostic Test Case #25 / 25
📋 Dataset Ground Truth Label: Polyp (polyp)
------------------------------------------------------------
📤 MedGemma Predicted Output : Polyp
🎯 Matched Assessment Class  : Polyp (Match: True)


     FINAL MEDGEMMA ENDOSCOPY CLASSIFICATION METRICS     
 📂 Total Images Classified : 25
 ✅ Correct Predictions      : 25
 ❌ Incorrect Predictions    : 0
------------------------------------------------------------
 📈 CALCULATED ACCURACY RATE : 100.00%



## 5. Fine-tuning MedGemma for Endoscopy Classification

To improve the model's performance on our specific binary classification task (Normal vs. Polyp in endoscopy images), we will fine-tune the MedGemma model. This involves training the pre-trained model on a subset of our collected data.

First, we'll load the necessary libraries and the `AutoProcessor` and `AutoModelForCausalLM` from Hugging Face Transformers. We need `AutoModelForCausalLM` because MedGemma is a causal language model that takes image inputs.

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoProcessor, AutoModelForCausalLM
from sklearn.model_selection import train_test_split as sklearn_train_test_split

# Base data for fine-tuning
raw_fine_tuning_data = test_samples

# ==========================================
# 1. SPLIT DATA & CONVERT TO HF DATASET
# ==========================================
# Split the raw Python list directly to avoid indexing conflicts with HF Dataset objects
train_samples, val_samples = sklearn_train_test_split(
    raw_fine_tuning_data,
    test_size=0.2,
    random_state=42,
    stratify=[item['label'] for item in raw_fine_tuning_data]  # Maintains the 50/50 split
)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_list(train_samples)
val_dataset = Dataset.from_list(val_samples)

print(f"✅ Training dataset size: {len(train_dataset)}")
print(f"✅ Validation dataset size: {len(val_dataset)}")


# ==========================================
# 2. LOAD PROCESSOR & MODEL
# ==========================================
print("\nLoading processor and model for fine-tuning...")

# Replace HF_TOKEN with your actual token string or environment variable if needed
processor = AutoProcessor.from_pretrained("google/medgemma-1.5-4b-it", token=hf_token)

model = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-1.5-4b-it",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

# Fix padding for generation/batching without corrupting the EOS token
if processor.tokenizer.pad_token is None:
    processor.tokenizer.add_special_tokens({"pad_token": "<pad>"})
    model.resize_token_embeddings(len(processor.tokenizer))

# MedGemma expects padding on the left side during causal generation tasks
processor.tokenizer.padding_side = "left"

✅ Training dataset size: 20
✅ Validation dataset size: 5

Loading processor and model for fine-tuning...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

### Preprocessing the Dataset

For fine-tuning a Vision-Language Model like MedGemma, we need a preprocessing function that handles both image and text inputs. This function will:

1.  Extract the image and raw label from each example.
2.  Convert the raw label into our target classification (`"Normal"` or `"Polyp"`).
3.  Construct the prompt in the conversational format expected by MedGemma, appending the ground truth label to the prompt as the target output for the model to learn.
4.  Use the `processor` to tokenize the text and prepare the image (e.g., resize, normalize). The `labels` for the causal language model will be derived from the `input_ids` with appropriate masking for loss calculation.

In [ ]:
def preprocess_function_batched(examples):
    images = examples['image']
    raw_labels = examples['label']

    ground_truths = ["Normal" if "normal" in str(label).lower() else "Polyp" for label in raw_labels]

    # We need to tokenize the prompts and the full text separately
    # to figure out exactly where the prompt ends and the label begins.
    batch_input_ids = []
    batch_attention_masks = []
    batch_labels = []
    batch_pixel_values = []

    for i in range(len(images)):
        # 1. Recreate the precise message structure
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": (
                        "Analyze this gastrointestinal endoscopy image. You must perform binary classification. "
                        "Determine if the tissue displays signs of a Polyp or if it is Normal mucosa.\n"
                        "Do not write descriptive findings or paragraphs. Your output must contain "
                        "exclusively one word, either 'Normal' or 'Polyp'."
                    )}
                ]
            }
        ]

        # Get the formatted string representation for the prompt ONLY
        prompt_text = processor.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Append the ground truth answer directly to it
        full_text = prompt_text + ground_truths[i]

        # 2. Process the individual multi-modal sample
        # Note: We do not pass padding=True here; we pad the whole batch together at the end.
        processed_sample = processor(
            text=full_text,
            images=images[i],
            return_tensors="pt",
            truncation=True
        )

        # Tokenize prompt ONLY to find its length
        processed_prompt = processor(
            text=prompt_text,
            images=images[i],
            return_tensors="pt",
            truncation=True
        )

        input_ids = processed_sample["input_ids"].squeeze(0)
        attention_mask = processed_sample["attention_mask"].squeeze(0)
        prompt_len = processed_prompt["input_ids"].shape[1]

        # 3. Create the labels mask
        # Clone input_ids, then overwrite everything belonging to the prompt with -100
        labels = input_ids.clone()
        labels[:prompt_len] = -100

        # Save to batch collectors
        batch_input_ids.append(input_ids)
        batch_attention_masks.append(attention_mask)
        batch_labels.append(labels)
        batch_pixel_values.append(processed_sample["pixel_values"].squeeze(0))

    # 4. Manual batch padding (Right padding is standard for training loss calculation)
    # Using the token ID we added in the previous cell step
    pad_token_id = processor.tokenizer.pad_token_id

    # Pad all sequences to the longest sequence in this specific batch
    max_len = max(len(ids) for ids in batch_input_ids)

    final_input_ids = []
    final_attention_masks = []
    final_labels = []

    for i in range(len(batch_input_ids)):
        pad_len = max_len - len(batch_input_ids[i])

        # Pad input_ids with pad_token_id
        final_input_ids.append(
            torch.cat([batch_input_ids[i], torch.full((pad_len,), pad_token_id, dtype=torch.long)])
        )
        # Pad attention mask with 0 (ignore)
        final_attention_masks.append(
            torch.cat([batch_attention_masks[i], torch.zeros(pad_len, dtype=torch.long)])
        )
        # Pad labels with -100 (ignore in loss calculation)
        final_labels.append(
            torch.cat([batch_labels[i], torch.full((pad_len,), -100, dtype=torch.long)])
        )

    return {
        "input_ids": torch.stack(final_input_ids),
        "attention_mask": torch.stack(final_attention_masks),
        "labels": torch.stack(final_labels),
        "pixel_values": torch.stack(batch_pixel_values)
    }

# Apply the preprocessing function to both training and validation datasets
print("\nPreprocessing training data...")
processed_train_dataset = train_dataset.map(preprocess_function_batched, batched=True, remove_columns=['image', 'label'])
print("Preprocessing validation data...")
processed_val_dataset = val_dataset.map(preprocess_function_batched, batched=True, remove_columns=['image', 'label'])

### Training the Model

We'll use Hugging Face's `Trainer` API, which simplifies the training loop considerably. We need to define `TrainingArguments` (e.g., output directory, learning rate, batch size, number of epochs) and then initialize and run the `Trainer`.

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers.training_args import EvaluationStrategy
from transformers.trainer_utils import IntervalStrategy as SaveStrategy # Corrected import for SaveStrategy
import torch

# 4. Define training arguments
training_args = TrainingArguments(
    output_dir="./fine_tuned_medgemma",
    num_train_epochs=3, # A small number of epochs for demonstration
    per_device_train_batch_size=2, # Small batch size due to potential GPU memory constraints
    per_device_eval_batch_size=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    evaluation_strategy=EvaluationStrategy.EPOCH, # Evaluate at the end of each epoch
    logging_dir="./logs",
    logging_steps=10,
    save_strategy=SaveStrategy.EPOCH, # Save model at the end of each epoch
    load_best_model_at_end=True, # Load the best model found during training at the end
    report_to="none", # Disable reporting to external services like Weights & Biases
    gradient_accumulation_steps=4, # Accumulate gradients to simulate a larger effective batch size
    fp16=torch.cuda.is_available(), # Use mixed precision (half-precision floats) if GPU is available
    remove_unused_columns=False # Important to keep pixel_values if present in the batch
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_train_dataset,
    eval_dataset=processed_val_dataset,
    tokenizer=processor.tokenizer,
)

# 6. Train the model
print("\nStarting fine-tuning...")
trainer.train()
print("\nFine-tuning complete!")

# Save the fine-tuned model and processor
model.save_pretrained("./fine_tuned_medgemma") # Save to the main directory
processor.save_pretrained("./fine_tuned_medgemma") # Save to the main directory
print("Fine-tuned model and processor saved to ./fine_tuned_medgemma")

ImportError: cannot import name 'EvaluationStrategy' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

### 6. Validating the Fine-tuned Model

After fine-tuning, we'll load the newly trained model and processor to perform inference on the validation set. This will allow us to compare its predictions against the ground truth labels and calculate the accuracy, demonstrating the impact of fine-tuning.

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM, pipeline
import torch

# Load the fine-tuned model for inference
print("\nLoading fine-tuned model for validation...")
fine_tuned_processor = AutoProcessor.from_pretrained("./fine_tuned_medgemma", token=hf_token)
fine_tuned_model = AutoModelForCausalLM.from_pretrained(
    "./fine_tuned_medgemma",
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token
)

# Create an inference pipeline from the fine-tuned model
fine_tuned_vqa_pipeline = pipeline(
    "image-text-to-text",
    model=fine_tuned_model,
    tokenizer=fine_tuned_processor.tokenizer,
    image_processor=fine_tuned_processor.image_processor,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Validate the fine-tuned model on the previously defined validation set
print("\n🔬 Validating fine-tuned model on the validation set...")

vqa_results_fine_tuned = []
correct_predictions_fine_tuned = 0

for idx, item in enumerate(val_samples, 1): # Iterate over the raw validation data, using val_samples not val_data
    pil_image = item['image']
    raw_label = item.get('label', 'Unknown')
    ground_truth = "Normal" if "normal" in raw_label.lower() else "Polyp"

    if pil_image.mode != 'RGB':
        pil_image = pil_image.convert('RGB')

    print(f"{'='*60}")
    print(f"🏃 Fine-tuned Diagnostic Test Case #{idx} / {len(val_samples)}")
    print(f"📋 Dataset Ground Truth Label: {ground_truth} ({raw_label})")
    print(f"{'-'*60}")

    # Structure the prompt for inference with the fine-tuned model
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": pil_image
                },
                {
                    "type": "text",
                    "text": (
                        "Analyze this gastrointestinal endoscopy image. You must perform binary classification. "
                        "Determine if the tissue displays signs of a Polyp or if it is Normal mucosa.\n"
                        "Do not write descriptive findings or paragraphs. Your output must contain "
                        "exclusively one word, either 'Normal' or 'Polyp'."
                    )
                }
            ]
        }
    ]

    # Generate prediction
    output = fine_tuned_vqa_pipeline(
        text=messages,
        max_new_tokens=10 # Keep generation short for classification token
    )

    chat_history = output[0]['generated_text']
    generated_assessment = chat_history[-1]['content'].strip()

    # Parse the generated output to match our strict classification
    prediction = "Unknown"
    lower_assessment = generated_assessment.lower()
    if "polyp" in lower_assessment:
        prediction = "Polyp"
    elif "normal" in lower_assessment:
        prediction = "Normal"

    # Check if the prediction is correct
    is_correct = (prediction == ground_truth)
    if is_correct:
        correct_predictions_fine_tuned += 1

    print(f"📤 Fine-tuned MedGemma Predicted Output : {generated_assessment}")
    print(f"🎯 Matched Assessment Class            : {prediction} (Match: {is_correct})")
    print(f"{'='*60}\n")

    # Store results
    case_record = {
        "test_case_number": idx,
        "ground_truth_label": ground_truth,
        "raw_label_name": raw_label,
        "medgemma_raw_output": generated_assessment,
        "parsed_prediction": prediction,
        "is_correct": is_correct
    }

    vqa_results_fine_tuned.append(case_record)

# Calculate and print final accuracy for the fine-tuned model
total_evaluated_fine_tuned = len(vqa_results_fine_tuned)
accuracy_percentage_fine_tuned = (correct_predictions_fine_tuned / total_evaluated_fine_tuned) * 100 if total_evaluated_fine_tuned > 0 else 0

print(f"\n{'='*60}")
print(f"     FINAL FINE-TUNED MEDGEMMA ENDOSCOPY CLASSIFICATION METRICS     ")
print(f"{'='*60}")
print(f" 📂 Total Images Classified : {total_evaluated_fine_tuned}")
print(f" ✅ Correct Predictions      : {correct_predictions_fine_tuned}")
print(f" ❌ Incorrect Predictions    : {total_evaluated_fine_tuned - correct_predictions_fine_tuned}")
print(f"{'-'*60}")
print(f" 📈 CALCULATED ACCURACY RATE : {accuracy_percentage_fine_tuned:.2f}%")
print(f"{'='*60}")


Loading fine-tuned model for validation...


OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './fine_tuned_medgemma'.

The fine-tuning process is complete, and the model has been evaluated on the validation set. The `CALCULATED ACCURACY RATE` above indicates the performance of the fine-tuned model on this specific task.

In [ ]:
import pandas as pd

df = pd.read_json(output_vqa_path)
# This displays a clean matrix of your labels alongside the clinical text
print(df[['test_case_number', 'ground_truth_label', 'medgemma_raw_output','parsed_prediction','is_correct']])

In [ ]:
print(f"The accuracy of the predictions is: {accuracy_percentage:.2f}%")